# Person A — Notebook 4 FAST (GPT-2-XL): Circuit Comparison
**CS 590NN | Amogh | Apr 19 — FAST GPT-2-XL arm**

Fast version of NB4 v2 on GPT-2-XL. 3-method comparison (ACDC, ROME-trace, Random).

**Speed config:**
- **N_SAMPLES = 20** (was 50)
- **End-of-loop KL measurement only**
- **n_steps = 5** (single operating point, not ablation)

**Runtime:** ~20-30 min on H100.


### 4.0 Install

In [ ]:
import subprocess, sys, os
def pip(args): subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + args)
pip(["numpy==1.26.4"])
pip(["transformer-lens", "transformers", "datasets", "accelerate", "einops", "jaxtyping", "matplotlib"])
print("Done. Restarting runtime...")
os.kill(os.getpid(), 9)


### 4.1 Imports

In [ ]:
import torch, json, re, requests, random
from pathlib import Path
from typing import List, Dict, Tuple, Optional
from dataclasses import dataclass, field
from transformer_lens import HookedTransformer
import matplotlib.pyplot as plt
import numpy as np

# Fail fast if GPU is missing — avoids wasting 1 min downloading 6 GB to CPU
assert torch.cuda.is_available(), (
    "CUDA not available. Go to Runtime → Change runtime type → Hardware accelerator → "
    "GPU (H100 if possible). Then Disconnect → Reconnect and re-run."
)

DEVICE = "cuda"
print(f"GPU: {torch.cuda.get_device_name(0)}")

torch.manual_seed(42); random.seed(42)
RESULTS_DIR = Path("results"); FIGS_DIR = Path("figures")
RESULTS_DIR.mkdir(exist_ok=True); FIGS_DIR.mkdir(exist_ok=True)


### 4.2 Load GPT-2-XL

In [ ]:
MODEL_NAME = "gpt2-xl"
model = HookedTransformer.from_pretrained(
    MODEL_NAME, center_unembed=True, center_writing_weights=False,
    fold_ln=True, refactor_factored_attn_matrices=False, device=DEVICE,
)
model.set_use_attn_result(True)
model.eval()
model.tokenizer.pad_token = model.tokenizer.eos_token
print(f"Loaded: {MODEL_NAME} | {model.cfg.n_layers} layers")


### 4.3 Upload ACDC Circuit Log

In [ ]:
from google.colab import files
print("Upload week2_circuit_log_gpt2xl.json (from NB1 FAST)...")
uploaded = files.upload()
with open(next(iter(uploaded.keys()))) as f:
    acdc_log = json.load(f)
print(f"Loaded {len(acdc_log)} ACDC circuits. avg n_mlp={sum(e['n_mlp'] for e in acdc_log)/len(acdc_log):.2f}")


### 4.4 Load CounterFact (20 samples)

In [ ]:
@dataclass
class EditSample:
    prompt: str; target_new: str; target_true: str; subject: str = ""
    related_prompts: List[str] = field(default_factory=list)

raw_data = requests.get("https://rome.baulab.info/data/dsets/counterfact.json", timeout=60).json()
N_SAMPLES = 20

def parse(raw):
    return EditSample(
        prompt=raw["requested_rewrite"]["prompt"].format(raw["requested_rewrite"]["subject"]),
        target_new=" " + raw["requested_rewrite"]["target_new"]["str"],
        target_true=" " + raw["requested_rewrite"]["target_true"]["str"],
        subject=raw["requested_rewrite"]["subject"],
    )

cf_samples = [parse(raw_data[i]) for i in range(N_SAMPLES)]
assert len(cf_samples) == len(acdc_log)


### 4.5 ROME-Trace (top-K=5 per sample)

In [ ]:
NOISE_SIGMA = 0.1
TOP_K_TRACE = 5

def find_subject_range(model, prompt, subject):
    full_tok = model.to_tokens(prompt)[0].tolist()
    pre_idx = prompt.find(subject)
    if pre_idx < 0:
        return max(0, len(full_tok) - 3), len(full_tok)
    prefix = prompt[:pre_idx]
    pre_tok = model.to_tokens(prefix)[0].tolist()
    start = len(pre_tok)
    sub_tok = model.to_tokens(prompt[:pre_idx + len(subject)])[0].tolist()
    return start, len(sub_tok)

def causal_trace(model, sample, n_noise_samples=3):
    tokens = model.to_tokens(sample.prompt)
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0, 0].item()
    s_start, s_end = find_subject_range(model, sample.prompt, sample.subject)

    with torch.no_grad():
        clean_logits, clean_cache = model.run_with_cache(tokens)
    clean_p = torch.softmax(clean_logits[0, -1, :], dim=-1)[new_id].item()

    def noise_hook(name):
        def fn(emb, hook):
            noise = torch.randn_like(emb[:, s_start:s_end, :]) * NOISE_SIGMA
            emb[:, s_start:s_end, :] = emb[:, s_start:s_end, :] + noise
            return emb
        return fn

    corrupt_p = 0.0
    for _ in range(n_noise_samples):
        with torch.no_grad():
            cl = model.run_with_hooks(tokens, fwd_hooks=[("hook_embed", noise_hook("hook_embed"))])
        corrupt_p += torch.softmax(cl[0, -1, :], dim=-1)[new_id].item()
    corrupt_p /= n_noise_samples

    indirect = {}
    restore_pos = s_end - 1
    for L in range(model.cfg.n_layers):
        hn = f"blocks.{L}.hook_mlp_out"
        clean_act = clean_cache[hn][:, restore_pos:restore_pos+1, :].clone()
        def restore_hook(act=clean_act, pos=restore_pos):
            def fn(v, hook):
                v[:, pos:pos+1, :] = act
                return v
            return fn
        torch.manual_seed(42)
        p_avg = 0.0
        for _ in range(n_noise_samples):
            with torch.no_grad():
                rl = model.run_with_hooks(tokens, fwd_hooks=[
                    ("hook_embed", noise_hook("hook_embed")),
                    (hn, restore_hook()),
                ])
            p_avg += torch.softmax(rl[0, -1, :], dim=-1)[new_id].item()
        p_avg /= n_noise_samples
        indirect[L] = p_avg - corrupt_p

    del clean_cache, clean_logits
    torch.cuda.empty_cache()
    return indirect, clean_p, corrupt_p

print(f"Running ROME-trace on {N_SAMPLES} GPT-2-XL samples...")
trace_circuits = []
for i, s in enumerate(cf_samples):
    try:
        effects, _, _ = causal_trace(model, s)
        top_layers = sorted(effects.items(), key=lambda x: -x[1])[:TOP_K_TRACE]
        trace_circuits.append({
            "idx": i, "mlp_layers": sorted([L for L, _ in top_layers]),
            "attn_heads": [], "n_mlp": TOP_K_TRACE, "n_attn": 0,
            "top_effects": [(int(L), round(e, 4)) for L, e in top_layers],
        })
        if i % 5 == 0:
            free = torch.cuda.mem_get_info()[0] / 1e9
            print(f"  [{i+1:>2}/{N_SAMPLES}]  top=[{','.join(str(L) for L,_ in top_layers)}]  gpu_free={free:.1f}GB")
    except Exception as e:
        print(f"  [{i+1:>2}/{N_SAMPLES}]  FAILED: {type(e).__name__}: {str(e)[:80]}")
        mid = model.cfg.n_layers // 2
        trace_circuits.append({
            "idx": i, "mlp_layers": list(range(mid-2, mid+3)),
            "attn_heads": [], "n_mlp": 5, "n_attn": 0, "trace_failed": True,
        })

with open(RESULTS_DIR / "week6_rome_trace_circuits_gpt2xl.json", "w") as f:
    json.dump(trace_circuits, f, indent=2)


### 4.6 Random Circuits

In [ ]:
random.seed(42)
random_circuits = []
for i, e in enumerate(acdc_log):
    K = e["n_mlp"]
    layers = sorted(random.sample(range(model.cfg.n_layers), min(K, model.cfg.n_layers)))
    random_circuits.append({
        "idx": i, "mlp_layers": layers, "attn_heads": [],
        "n_mlp": len(layers), "n_attn": 0,
    })
print(f"Random avg n_mlp = {sum(c['n_mlp'] for c in random_circuits)/len(random_circuits):.2f}")


### 4.7 Edit Functions (same FAST kernel as NB3)

In [ ]:
NEUTRAL_PROMPTS = [
    "The sum of two and three is", "Twelve divided by four equals",
    "The square root of nine is", "Ten times ten equals",
    "The capital of Japan is", "The largest ocean on Earth is the",
    "Mount Everest is located in the", "The Amazon River flows through",
    "Water boils at one hundred degrees", "The chemical symbol for gold is",
    "Plants produce oxygen through a process called", "The Earth orbits the",
    "A week contains seven", "The primary colors are red, blue, and",
    "Humans have two lungs and one", "A triangle has three",
    "The opposite of hot is", "A baby cat is called a",
    "The past tense of run is", "A group of fish is called a",
    "The Industrial Revolution began in the eighteenth",
    "The Renaissance was a period of cultural",
    "World War Two ended in the year", "The Wright Brothers invented the",
    "A computer's main processor is the", "The world wide web was invented by",
    "An algorithm is a sequence of", "The unit of electrical resistance is the",
    "Roses are typically red while violets are",
    "The sky appears blue because of light",
    "A year contains twelve", "The freezing point of water is zero degrees",
]

def restore_weights(model, state):
    with torch.no_grad():
        for name, param in model.named_parameters():
            param.copy_(state[name])
    torch.cuda.empty_cache()

def cache_reference_logits(model, prompts):
    cache = []
    model.eval()
    with torch.no_grad():
        for p in prompts:
            tokens = model.to_tokens(p)
            logits = model(tokens)
            cache.append((tokens, torch.log_softmax(logits[0, -1, :], dim=-1).detach().clone()))
    return cache

def get_circuit_params(model, circuit_attn, circuit_mlp):
    params = []
    for (layer, head) in circuit_attn:
        try: params.append(model.blocks[layer].attn.W_O)
        except Exception: pass
    for layer in circuit_mlp:
        try:
            params.append(model.blocks[layer].mlp.W_in)
            params.append(model.blocks[layer].mlp.W_out)
        except Exception: pass
    return params

def contrastive_loss(model, sample):
    tokens = model.to_tokens(sample.prompt)
    new_id = model.to_tokens(sample.target_new, prepend_bos=False)[0, 0]
    true_id = model.to_tokens(sample.target_true, prepend_bos=False)[0, 0]
    logits = model(tokens)
    lp = torch.nn.functional.log_softmax(logits[0, -1, :], dim=-1)
    loss = -lp[new_id] + lp[true_id]
    return loss, lp[new_id].exp().item(), lp[true_id].exp().item()

def kl_loss_neutral(model, ref_cache):
    total = 0.0
    for tokens, ref_lp in ref_cache:
        logits = model(tokens)
        edit_lp = torch.log_softmax(logits[0, -1, :], dim=-1)
        total = total + (ref_lp.exp() * (ref_lp - edit_lp)).sum()
    return total / len(ref_cache)

def kl_loss_neutral_eval(model, ref_cache):
    total = 0.0
    model.eval()
    with torch.no_grad():
        for tokens, ref_lp in ref_cache:
            logits = model(tokens)
            edit_lp = torch.log_softmax(logits[0, -1, :], dim=-1)
            total += (ref_lp.exp() * (ref_lp - edit_lp)).sum().item()
    return total / len(ref_cache)

def run_edit_fast(model, sample, circuit_attn, circuit_mlp, n_steps,
                  lr=5e-3, beta_kl=0.1, grad_clip=1.0, ref_cache=None):
    params = get_circuit_params(model, circuit_attn, circuit_mlp)
    if not params:
        params = [p for layer in model.blocks for p in [layer.mlp.W_in, layer.mlp.W_out]]
    for p in model.parameters(): p.requires_grad_(False)
    for p in params: p.requires_grad_(True)
    optimizer = torch.optim.Adam(params, lr=lr)

    model.eval()
    with torch.no_grad():
        _, baseline, _ = contrastive_loss(model, sample)
    kl_initial = kl_loss_neutral_eval(model, ref_cache) if ref_cache is not None else 0.0

    for _ in range(n_steps):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss, _, _ = contrastive_loss(model, sample)
        if ref_cache is not None and beta_kl > 0:
            kl = kl_loss_neutral(model, ref_cache)
            loss = loss + beta_kl * kl
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, max_norm=grad_clip)
        optimizer.step()
        del loss

    torch.cuda.empty_cache()
    model.eval()
    with torch.no_grad():
        _, final_p_new, final_p_true = contrastive_loss(model, sample)
    kl_final = kl_loss_neutral_eval(model, ref_cache) if ref_cache is not None else 0.0
    for p in model.parameters(): p.requires_grad_(True)
    return {
        "edit_success": final_p_new, "baseline_prob": baseline, "final_p_true": final_p_true,
        "kl_initial": kl_initial, "kl_final": kl_final, "delta_kl": kl_final - kl_initial,
        "status": "ok",
    }

print("Snapshotting GPT-2-XL weights to CPU...")
original_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
torch.cuda.empty_cache()


### 4.8 Comparison Loop (3 methods × 20 samples × n_steps=5)

In [ ]:
N_STEPS = 5; LR = 5e-3; BETA_KL = 0.1; GRAD_CLIP = 1.0

methods = {
    "OurMethod_ACDC_gpt2xl":      acdc_log,
    "OurMethod_ROMEtrace_gpt2xl": trace_circuits,
    "OurMethod_Random_gpt2xl":    random_circuits,
}
comparison_results = {m: [] for m in methods}

for method_name, circuits in methods.items():
    print(f"\n=== {method_name} ===")
    n_failed = 0
    for i in range(N_SAMPLES):
        try:
            restore_weights(model, original_state)
            ref_cache = cache_reference_logits(model, NEUTRAL_PROMPTS)
            res = run_edit_fast(
                model, cf_samples[i],
                circuits[i].get("attn_heads", []),
                circuits[i]["mlp_layers"],
                n_steps=N_STEPS, lr=LR, beta_kl=BETA_KL,
                grad_clip=GRAD_CLIP, ref_cache=ref_cache,
            )
            res["idx"] = i; res["n_mlp"] = circuits[i]["n_mlp"]
            res["n_attn"] = circuits[i].get("n_attn", 0); res["n_steps"] = N_STEPS
            comparison_results[method_name].append(res)
            del ref_cache
            torch.cuda.empty_cache()
            if i % 5 == 0:
                free = torch.cuda.mem_get_info()[0] / 1e9
                print(f"  [{i+1:>2}/{N_SAMPLES}]  edit={res['edit_success']:.4f}  kl={res['kl_final']:.3f}  gpu_free={free:.1f}GB")
        except Exception as e:
            n_failed += 1
            print(f"  [{i+1:>2}/{N_SAMPLES}]  FAILED: {type(e).__name__}: {str(e)[:80]}")
            comparison_results[method_name].append({
                "idx": i, "n_steps": N_STEPS, "status": "failed",
                "error": f"{type(e).__name__}: {str(e)[:200]}",
                "edit_success": 0.0, "baseline_prob": 0.0, "final_p_true": 0.0,
                "kl_initial": 0.0, "kl_final": 0.0, "delta_kl": 0.0,
                "n_mlp": circuits[i]["n_mlp"], "n_attn": circuits[i].get("n_attn", 0),
            })
            torch.cuda.empty_cache()
    ok = [r for r in comparison_results[method_name] if r.get("status") == "ok"]
    if ok:
        avg_e = sum(r["edit_success"] for r in ok) / len(ok)
        avg_k = sum(r["kl_final"] for r in ok) / len(ok)
        flipped = sum(1 for r in ok if r["edit_success"] > 0.5)
        print(f"  SUMMARY  edit={avg_e:.4f}  kl={avg_k:.3f}  flip={flipped}/{len(ok)}  failed={n_failed}")

restore_weights(model, original_state)


### 4.9 Figures + Output

In [ ]:
COLORS = {"OurMethod_ACDC_gpt2xl": "C0", "OurMethod_ROMEtrace_gpt2xl": "C2", "OurMethod_Random_gpt2xl": "C3"}
LABELS = {"OurMethod_ACDC_gpt2xl": "ACDC", "OurMethod_ROMEtrace_gpt2xl": "ROME-trace", "OurMethod_Random_gpt2xl": "Random"}

# Pareto scatter
fig, ax = plt.subplots(figsize=(7, 5))
for method, results in comparison_results.items():
    ok = [r for r in results if r.get("status") == "ok"]
    if not ok: continue
    es = [r["edit_success"] for r in ok]
    kl = [r["kl_final"] for r in ok]
    ax.scatter(kl, es, c=COLORS[method], alpha=0.5, s=40, label=LABELS[method])
    mu_kl = sum(kl)/len(kl); mu_es = sum(es)/len(es)
    ax.scatter([mu_kl], [mu_es], c=COLORS[method], s=260, marker="X",
               edgecolors="black", linewidths=1.5,
               label=f"{LABELS[method]} mean (kl={mu_kl:.1f}, es={mu_es:.2f})")
ax.set_xlabel("KL_final"); ax.set_ylabel("edit_success")
ax.set_title(f"GPT-2-XL Pareto ({N_STEPS}-step, β={BETA_KL}, N={N_SAMPLES})")
ax.set_ylim(-0.05, 1.08); ax.grid(alpha=0.3); ax.legend(fontsize=8, loc="lower right")
fig.tight_layout(); fig.savefig(FIGS_DIR / "fig_pareto_gpt2xl.png", dpi=160); plt.show()

# Bars
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
mlist = list(methods.keys())
mu_es, sd_es, mu_kl, sd_kl = [], [], [], []
for m in mlist:
    ok = [r for r in comparison_results[m] if r.get("status") == "ok"]
    es = np.array([r["edit_success"] for r in ok])
    kl = np.array([r["kl_final"] for r in ok])
    mu_es.append(es.mean()); sd_es.append(es.std())
    mu_kl.append(kl.mean()); sd_kl.append(kl.std())
x = np.arange(len(mlist))
ax1.bar(x, mu_es, yerr=sd_es, color=[COLORS[m] for m in mlist], capsize=6, alpha=0.8)
ax1.set_xticks(x); ax1.set_xticklabels([LABELS[m] for m in mlist])
ax1.set_ylabel("edit_success"); ax1.set_ylim(0, 1.1)
ax1.set_title(f"GPT-2-XL (N={N_SAMPLES}): Edit success"); ax1.grid(axis="y", alpha=0.3)
for xi, v in zip(x, mu_es): ax1.text(xi, v + 0.02, f"{v:.3f}", ha="center", fontsize=10)
ax2.bar(x, mu_kl, yerr=sd_kl, color=[COLORS[m] for m in mlist], capsize=6, alpha=0.8)
ax2.set_xticks(x); ax2.set_xticklabels([LABELS[m] for m in mlist])
ax2.set_ylabel("KL_final"); ax2.set_title("GPT-2-XL: KL drift")
ax2.grid(axis="y", alpha=0.3)
for xi, v in zip(x, mu_kl): ax2.text(xi, v + 0.5, f"{v:.2f}", ha="center", fontsize=10)
fig.tight_layout(); fig.savefig(FIGS_DIR / "fig_method_bars_gpt2xl.png", dpi=160); plt.show()


rows, summary = [], {}
for method_name, results in comparison_results.items():
    for r in results:
        rows.append({
            "method": method_name, "model": MODEL_NAME, "idx": r["idx"],
            "n_steps": r["n_steps"],
            "edit_success": round(r["edit_success"], 4),
            "baseline_prob": round(r["baseline_prob"], 4),
            "over_extinction": None,
            "kl_initial": round(r["kl_initial"], 4),
            "kl_final":   round(r["kl_final"],   4),
            "delta_kl":   round(r["delta_kl"],   4),
            "kl_active": True, "beta_kl": BETA_KL, "grad_clip": GRAD_CLIP,
            "n_mlp": r["n_mlp"], "n_attn": r["n_attn"],
            "status": r.get("status", "ok"), "error": r.get("error", None),
        })
    ok = [r for r in results if r.get("status") == "ok"]
    if ok:
        summary[method_name] = {
            "avg_edit_success": round(sum(r["edit_success"] for r in ok) / len(ok), 4),
            "avg_kl_final":     round(sum(r["kl_final"] for r in ok) / len(ok), 4),
            "avg_n_mlp":        round(sum(r["n_mlp"] for r in ok) / len(ok), 2),
            "pct_flipped":      round(sum(1 for r in ok if r["edit_success"] > 0.5) / len(ok), 4),
            "n_ok": len(ok), "n_failed": len(results) - len(ok),
        }

with open(RESULTS_DIR / "week6_comparison_output_gpt2xl.json", "w") as f:
    json.dump({
        "experiment": "circuit_comparison_expt2_gpt2xl",
        "model": MODEL_NAME, "n_steps": N_STEPS,
        "n_samples": N_SAMPLES,
        "beta_kl": BETA_KL, "grad_clip": GRAD_CLIP,
        "neutral_set_size": len(NEUTRAL_PROMPTS),
        "notebook_version": "fast_gpt2xl",
        "rows": rows, "summary": summary,
    }, f, indent=2)


import shutil
from datetime import datetime
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
with open(RESULTS_DIR / f"summary_nb4_gpt2xl_FAST_{timestamp}.json", "w") as f:
    json.dump({
        "author": "Amogh",
        "notebook": "Notebook 4 FAST (GPT-2-XL) — Circuit Comparison",
        "model": MODEL_NAME, "timestamp": timestamp,
        "n_samples": N_SAMPLES, "n_steps": N_STEPS,
        "fast_version": True,
        "beta_kl": BETA_KL, "grad_clip": GRAD_CLIP,
        "methods": list(methods.keys()), "summary": summary,
    }, f, indent=2)

all_dir = Path("zip_contents")
all_dir.mkdir(exist_ok=True)
for p in RESULTS_DIR.glob("*"): shutil.copy2(p, all_dir / p.name)
for p in FIGS_DIR.glob("*"):    shutil.copy2(p, all_dir / p.name)
zip_path = f"/content/PersonA_Notebook4_gpt2xl_FAST_{timestamp}"
shutil.make_archive(zip_path, "zip", all_dir)

print("=" * 72)
print(f"  NB4 FAST (GPT-2-XL) — {timestamp}")
print("=" * 72)
print(f"  {'Method':<32}  {'edit':>6}  {'flip%':>6}  {'kl':>7}  {'n_mlp':>6}")
for m, s in summary.items():
    print(f"  {m:<32}  {s['avg_edit_success']:>6.4f}  "
          f"{100*s['pct_flipped']:>5.1f}%  {s['avg_kl_final']:>7.3f}  {s['avg_n_mlp']:>6.2f}")
print(f"\n  Download: {zip_path}.zip")

from google.colab import files
files.download(f"{zip_path}.zip")
